# RAG System with RAGBench - Using Ingestion Layer and Strategy Classes

This notebook demonstrates the complete RAG pipeline using:
- Ingestion Layer for loading and parsing RAGBench datasets
- Strategy Factory for creating strategies
- RAG Config for configuration
- Complete RAG Pipeline for orchestration

## 1. Setup & Dependencies

In [ ]:
!pip install datasets faiss-cpu sentence-transformers torch groq python-dotenv nltk -q


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## 2. Imports

In [2]:
import os
import json
import pandas as pd
from typing import List, Dict, Any
from dotenv import load_dotenv
from groq import Groq
import numpy as np

# Load environment variables
load_dotenv(override=True)

# Get API keys
HF_TOKEN = os.getenv("HF_TOKEN")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print(f"HuggingFace token loaded: {bool(HF_TOKEN)}")
print(f"Groq API key loaded: {bool(GROQ_API_KEY)}")

HuggingFace token loaded: True
Groq API key loaded: True


## 3. Initialize Groq Client

In [3]:
# Initialize Groq client
client = Groq(api_key=GROQ_API_KEY)
print("✅ Groq client initialized!")

# Test connection
test_response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Say 'Ready' in one word."}],
    max_tokens=10
)
print(f"Test response: {test_response.choices[0].message.content}")

✅ Groq client initialized!
Test response: Ready.


## 4. Load Configuration

In [4]:
from rag.config.loader import ConfigLoader
from rag.config.examples import config_high_quality

# Use high quality config with reranking
config = config_high_quality

print("📋 Configuration:")
print(f"  Chunking: {config.chunking.type.value}")
print(f"  Embedding: {config.embedding.type.value}")
print(f"  Retrieval: {config.retrieval.type.value}")
print(f"  Generation: {config.generation.type.value} ({config.generation.model})")
print(f"  Evaluation: {config.evaluation.type.value}")

📋 Configuration:
  Chunking: sentence
  Embedding: bge
  Retrieval: dense_rerank
  Generation: groq (llama-3.1-70b-versatile)
  Evaluation: trace


## 5. Load Data Using Data Loader

In [ ]:
## 5. Load Data Using Ingestion Layer
from ingestion import (
    RAGBenchCovidQALoader,
    ParserFactory,
    ParserType,
    DataProcessor,
    DatasetLoadingConfig
)

# Create loader for CovidQA dataset
dataset_loading_config = DatasetLoadingConfig(limit=246)  # Full dataset
loader = RAGBenchCovidQALoader(
    split="test",
    config=dataset_loading_config
)

# Load raw data
print("📥 Loading RAGBench CovidQA dataset...")
raw_data = loader.load()
print(f"✅ Loaded {len(raw_data)} samples\n")

# Get dataset info
info = loader.info()
print(f"Dataset Info:")
print(f"  Source: {info['source']}")
print(f"  Dataset: {info['dataset_name']}")
print(f"  Split: {info['split']}")
print(f"  Samples: {info['num_samples']}")
print(f"  Keys: {len(info['keys'])} fields\n")

# Inspect first sample
first_sample = loader.load_sample(0)
print(f"First Sample:")
print(f"  Question: {first_sample['question'][:100]}...")
print(f"  Documents: {len(first_sample['documents'])}")
print(f"  Has ground truth scores: {bool(first_sample.get('relevance_score'))}")

/Users/adityanarayan/aiml-talentsprint/rag-capstone-project/real-world-rag-system/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📥 Loading RAGBench CovidQA dataset...
Loading RAGBench dataset: covidqa (test)...
Loaded 246 samples
✅ Loaded 246 samples

Dataset Info:
  Source: galileo-ai/ragbench
  Dataset: covidqa
  Split: test
  Samples: 246
  Keys: 26 fields

First Sample:
  Question: Which viruses may not cause prolonged inflammation due to strong induction of antiviral clearance?...
  Documents: 4
  Has ground truth scores: True


## 6. Parse Documents

In [3]:
## 6. Parse Documents Using Ingestion Layer
# Create parser strategy
print("📄 Creating parser strategy...")
parser = ParserFactory.create_parser(ParserType.TITLE_PASSAGE)
print(f"✅ Parser created: {parser.__class__.__name__}\n")

# Create data processor with parser strategy
processor = DataProcessor(parser_strategy=parser)

# Process dataset into canonical Document objects
print("📄 Processing documents...")
documents = processor.process_dataset(raw_data)
print(f"✅ Processed {len(documents)} documents\n")

# Print sample
sample_doc = documents[0]
print(f"Sample RAG Document:")
print(f"  Title: {sample_doc.title[:60]}...")
print(f"  Content: {sample_doc.content[:100]}...")
print(f"  Metadata: {sample_doc.metadata}")

📄 Creating parser strategy...
✅ Parser created: TitlePassageParser

📄 Processing documents...
✅ Processed 984 documents

Sample RAG Document:
  Title: Type I Interferon Receptor Deficiency in Dendritic Cells Fac...
  Content: successful treatment for HCV serves to circumvent the viral inhibition of IFN induction. Thus, HCV m...
  Metadata: {'doc_id': 'sample_0_doc_0', 'sample_index': 0, 'source': 'ragbench', 'parser_type': 'title_passage'}


## 7. Create Reranking Model

In [ ]:
from sentence_transformers import CrossEncoder

# The embedding model is created inside the pipeline from the RAG config
# (config.embedding.model_name). Only the reranker is built here and passed
# via `clients`, since it has no corresponding config field.

# Load reranking model
RERANKING_MODEL = "BAAI/bge-reranker-v2-m3"
print(f"Loading reranking model: {RERANKING_MODEL}")
rerank_model = CrossEncoder(RERANKING_MODEL)
print(f"✅ Loaded")

## 8. Initialize RAG Pipeline

In [ ]:
from examples.complete_pipeline_example import RAGPipeline

# Prepare clients
clients = {
    'groq': client,
    'reranker': rerank_model
}

# Create pipeline with config
print("🔧 Initializing RAG Pipeline...")
pipeline = RAGPipeline(config, clients)
print(f"✅ Pipeline initialized\n")

# Verify strategies were created
print("Strategies created:")
print(f"  Chunker: {pipeline.chunker.__class__.__name__}")
print(f"  Embedder: {pipeline.embedder.__class__.__name__}")
print(f"  Retriever: {pipeline.retriever.__class__.__name__}")
print(f"  Generator: {pipeline.generator.__class__.__name__}")
print(f"  Evaluator: {pipeline.evaluator.__class__.__name__}")

## 9. Build Vector Index

In [ ]:
# Build index - this will chunk, embed, and store
print("🏗️  Building vector index...")
pipeline.build_index(documents)
print(f"✅ Index built and ready for querying")

## 10. Test Single Query

In [ ]:
# Test with first sample
test_sample = raw_data[0]
test_query = test_sample['question']

print(f"❓ Test Query:")
print(f"   {test_query}\n")

# Run query through pipeline
result = pipeline.query(test_query, top_k=5)

# Display results
print(f"\n📊 Query Results:")
print(f"\nRetrieved Documents: {len(result['retrieved_docs'])}")
for i, doc in enumerate(result['retrieved_docs'][:3], 1):
    print(f"  {i}. {doc['metadata']['title'][:60]}...")

print(f"\n💬 Generated Response:")
print(result['response'][:300] + "...\n")

print(f"📈 TRACe Scores:")
scores = result['scores']
print(f"  Relevance:    {scores['relevance_score']:.4f}")
print(f"  Utilization:  {scores['utilization_score']:.4f}")
print(f"  Completeness: {scores['completeness_score']:.4f}")
print(f"  Adherence:    {scores['adherence_score']}")

## 11. Batch Evaluation

In [ ]:
# Evaluate on batch of samples
print("🔄 Running batch evaluation on first 10 samples...\n")

evaluation_results = []

for idx in range(min(10, len(raw_data))):
    sample = raw_data[idx]
    query = sample['question']
    
    # Get ground truth
    gt_relevance = sample.get('relevance_score', 0)
    gt_utilization = sample.get('utilization_score', 0)
    gt_completeness = sample.get('completeness_score', 0)
    gt_adherence = sample.get('adherence_score', False)
    
    # Run pipeline
    try:
        result = pipeline.query(query, top_k=5)
        scores = result['scores']
        
        evaluation_results.append({
            'Sample': idx,
            'Query': query[:50],
            'Our Relevance': scores['relevance_score'],
            'GT Relevance': gt_relevance,
            'Our Utilization': scores['utilization_score'],
            'GT Utilization': gt_utilization,
            'Our Completeness': scores['completeness_score'],
            'GT Completeness': gt_completeness,
            'Our Adherence': scores['adherence_score'],
            'GT Adherence': gt_adherence
        })
        print(f"✅ Sample {idx}: Relevance={scores['relevance_score']:.4f} | Adherence={scores['adherence_score']}")
    except Exception as e:
        print(f"⚠️  Sample {idx} failed: {str(e)[:50]}")

# Create results dataframe
df_results = pd.DataFrame(evaluation_results)
print(f"\n✅ Completed evaluation on {len(df_results)} samples")

## 12. Analysis of Results

In [ ]:
# Compare our scores with ground truth
print("📊 Comparison with Ground Truth\n")

print("Relevance Score:")
print(f"  Our avg:  {df_results['Our Relevance'].mean():.4f}")
print(f"  GT avg:   {df_results['GT Relevance'].mean():.4f}")
print(f"  Diff:     {abs(df_results['Our Relevance'].mean() - df_results['GT Relevance'].mean()):.4f}")

print("\nUtilization Score:")
print(f"  Our avg:  {df_results['Our Utilization'].mean():.4f}")
print(f"  GT avg:   {df_results['GT Utilization'].mean():.4f}")
print(f"  Diff:     {abs(df_results['Our Utilization'].mean() - df_results['GT Utilization'].mean()):.4f}")

print("\nCompleteness Score:")
print(f"  Our avg:  {df_results['Our Completeness'].mean():.4f}")
print(f"  GT avg:   {df_results['GT Completeness'].mean():.4f}")
print(f"  Diff:     {abs(df_results['Our Completeness'].mean() - df_results['GT Completeness'].mean()):.4f}")

print("\nAdherence (Exact Match):")
our_adherence = (df_results['Our Adherence'] == df_results['GT Adherence']).sum() / len(df_results)
print(f"  Match rate: {our_adherence:.2%}")

## 13. Interactive Querying

In [ ]:
# Try custom queries
custom_queries = [
    "What is COVID-19?",
    "How is the virus transmitted?",
    "What are the symptoms?"
]

print("🔍 Custom Query Examples\n")

for query in custom_queries:
    print(f"❓ Query: {query}")
    result = pipeline.query(query, top_k=3)
    print(f"💬 Answer: {result['response'][:200]}...\n")

## 14. Display Full Results Table

In [ ]:
# Show full results
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)

print("Full Evaluation Results:")
display(df_results[[
    'Sample', 'Query',
    'Our Relevance', 'GT Relevance',
    'Our Utilization', 'GT Utilization',
    'Our Completeness', 'GT Completeness',
    'Our Adherence', 'GT Adherence'
]])

## 15. Summary

In [ ]:
print("""
╔════════════════════════════════════════════════════════╗
║           RAG SYSTEM SUMMARY                            ║
╚════════════════════════════════════════════════════════╝

✅ Components Used:
   • Ingestion Layer - Loaded and parsed RAGBench dataset
   • RAGConfig - Configuration management
   • StrategyFactory - Created all strategies
   • RAGPipeline - Orchestrated the pipeline

📊 Pipeline Configuration:
""")

print(f"   Chunking: {config.chunking.type.value} ({config.chunking.max_words} words)")
print(f"   Embedding: {config.embedding.type.value} ({embedding_dim}D)")
print(f"   Retrieval: {config.retrieval.type.value} (initial_k={config.retrieval.initial_k})")
print(f"   Generation: {config.generation.type.value} ({config.generation.model})")
print(f"   Evaluation: {config.evaluation.type.value}")

print(f"""
📈 Results on {len(df_results)} samples:
   Relevance:    {df_results['Our Relevance'].mean():.4f} (GT: {df_results['GT Relevance'].mean():.4f})
   Utilization:  {df_results['Our Utilization'].mean():.4f} (GT: {df_results['GT Utilization'].mean():.4f})
   Completeness: {df_results['Our Completeness'].mean():.4f} (GT: {df_results['GT Completeness'].mean():.4f})
   Adherence:    {our_adherence:.2%} match rate

✨ Key Advantages:
   ✓ Clean, modular code
   ✓ Reusable components
   ✓ Configuration-driven
   ✓ Easy to extend
   ✓ Production-ready
""")